In [6]:
import sys
from pathlib import Path
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
import numpy as np
from qne.cascade import Key
from qne.cascade.sdc_runs import run_sdc_experiment, run_sdc_experiment_safe

def make_synthetic_pair(n_bits=1000, qber=0.02, seed=42):
    alice_key = Key(nr_bits=n_bits, seed=seed)
    bob_key = alice_key.copy()
    bob_key.apply_noise(bit_error_rate=qber, seed=seed + 1)
    return alice_key, bob_key


In [ ]:
from qne.cascade.sweep_utils import run_condition_sweep, summarize_outcomes


In [ ]:
# --- run all four conditions ---
alice_key, bob_key = make_synthetic_pair(n_bits=1000, qber=0.02, seed=42)

all_dfs = {}
summary_rows = []

conditions = [
    ("toeplitz_only", {"toeplitz_prob": 0.5}),
    ("final_key_only", {"final_key_prob": 0.5}),
    ("reconciliation_only", {"reconciliation_prob": 0.3}),
    ("combined", {"toeplitz_prob": 0.5, "final_key_prob": 0.5, "reconciliation_prob": 0.3}),
]

for label, kwargs in conditions:
    print(f"\nRunning {label}...")
    df_cond = run_condition_sweep(alice_key, bob_key, qber=0.02, label=label, n_runs=30, **kwargs)
    all_dfs[label] = df_cond
    summary_rows.append(summarize_outcomes(df_cond, label))

summary_df = pd.DataFrame(summary_rows)
print("\n\n=== FULL SUMMARY ===")
print(summary_df.to_string(index=False))

full_df = pd.concat(all_dfs.values(), ignore_index=True)
full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_fault_injection_results.csv"), index=False)
print("\nSaved -> results/sdc_fault_injection_results.csv")


In [ ]:
# --- dose-response sweep for Toeplitz-matrix, final-key, and
# reconciliation-state faults (reuses run_condition_sweep/summarize_outcomes
# defined above -- no need to redefine them) ---
summary_rows_dr = []
all_dfs_dr = {}

print("\n########## TOEPLITZ MATRIX FAULT SWEEP ##########")
toeplitz_dfs = []
for prob in [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]:
    label = f"toeplitz_{prob}"
    print(f"\nRunning {label}...")
    df_cond = run_condition_sweep(alice_key, bob_key, qber=0.02, label=label, n_runs=30, toeplitz_prob=prob)
    df_cond["fault_type"] = "Toeplitz-matrix"
    df_cond["prob"] = prob
    toeplitz_dfs.append(df_cond)
    summary_rows_dr.append(summarize_outcomes(df_cond, label))

print("\n########## FINAL KEY FAULT SWEEP ##########")
final_key_dfs = []
for prob in [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]:
    label = f"final_key_{prob}"
    print(f"\nRunning {label}...")
    df_cond = run_condition_sweep(alice_key, bob_key, qber=0.02, label=label, n_runs=30, final_key_prob=prob)
    df_cond["fault_type"] = "Final-key"
    df_cond["prob"] = prob
    final_key_dfs.append(df_cond)
    summary_rows_dr.append(summarize_outcomes(df_cond, label))

print("\n########## RECONCILIATION FAULT SWEEP ##########")
recon_dfs = []
for prob in [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]:
    label = f"reconciliation_{prob}"
    print(f"\nRunning {label}...")
    df_cond = run_condition_sweep(alice_key, bob_key, qber=0.02, label=label, n_runs=30, reconciliation_prob=prob)
    df_cond["reconciliation_prob"] = prob
    recon_dfs.append(df_cond)
    summary_rows_dr.append(summarize_outcomes(df_cond, label))

summary_df_dr = pd.DataFrame(summary_rows_dr)
print("\n\n=== FULL DOSE-RESPONSE SUMMARY ===")
print(summary_df_dr.to_string(index=False))

# Save under the EXACT filenames the analysis cells below actually load.
df_toeplitz_finalkey = pd.concat(toeplitz_dfs + final_key_dfs, ignore_index=True)
df_toeplitz_finalkey.to_csv(str(PROJECT_DIR / "results" / "sdc_toeplitz_finalkey_dose_response.csv"), index=False)
print("\nSaved -> results/sdc_toeplitz_finalkey_dose_response.csv")

df_reconciliation = pd.concat(recon_dfs, ignore_index=True)
df_reconciliation.to_csv(str(PROJECT_DIR / "results" / "sdc_reconciliation_dose_response.csv"), index=False)
print("Saved -> results/sdc_reconciliation_dose_response.csv")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Load saved results (now guaranteed to exist -- see cell 4 above) ---
all_data = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_toeplitz_hash_dose_response.csv"))

# --- TABLE 1: QBER vs. probability, per fault type ---
qber_table = (
    all_data.groupby(["fault_type", "prob"])["qber_after_reconciliation"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
print("=== QBER after reconciliation, by fault type and probability ===")
print(qber_table.to_string(index=False))

# --- TABLE 2: Mismatch rate vs. probability, per fault type ---
mismatch_table = (
    all_data.groupby(["fault_type", "prob"])
    .apply(lambda g: pd.Series({
        "mismatch_rate": (~g["keys_match"]).mean(),
        "n": len(g),
    }))
    .reset_index()
)
print("\n=== Key mismatch rate, by fault type and probability ===")
print(mismatch_table.to_string(index=False))

# --- FIGURE: two panels, QBER (flat) vs mismatch rate (climbing) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax1, ax2 = axes

for fault_type, group in all_data.groupby("fault_type"):
    g = group.groupby("prob")["qber_after_reconciliation"].mean().reset_index()
    ax1.plot(g["prob"], g["qber_after_reconciliation"] * 100, marker='o', label=fault_type)

ax1.set(xlabel="Fault probability", ylabel="QBER after reconciliation (%)",
        title="QBER: no signal of injected faults\n(protocol sees nothing wrong)")
ax1.set_xscale("log")
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(-1, 10)

for fault_type, group in all_data.groupby("fault_type"):
    g = group.groupby("prob").apply(lambda x: (~x["keys_match"]).mean()).reset_index(name="mismatch_rate")
    ax2.plot(g["prob"], g["mismatch_rate"] * 100, marker='o', label=fault_type)

ax2.set(xlabel="Fault probability", ylabel="Final key mismatch rate (%)",
        title="Final key: faults silently corrupt output")
ax2.set_xscale("log")
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle("Silent Data Corruption: undetectable based on QBER, visible in final-key agreement")
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "results" / "sdc_qber_vs_mismatch.png"), dpi=150)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Load reconciliation dose-response data (now guaranteed to exist) ---
df_recon = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_reconciliation_dose_response.csv"))

# --- Table: non-convergence rate by probability ---
recon_table = (
    df_recon.groupby("reconciliation_prob")
    .apply(lambda g: pd.Series({
        "non_convergent_rate": g["non_convergent"].mean(),
        "n": len(g),
    }))
    .reset_index()
)
print("=== Reconciliation-state fault: non-convergence rate by probability ===")
print(recon_table.to_string(index=False))

# --- Three-panel figure: QBER (flat), mismatch rate (toeplitz/hash), non-convergence (reconciliation) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax1, ax2, ax3 = axes

for fault_type, group in all_data.groupby("fault_type"):
    g = group.groupby("prob")["qber_after_reconciliation"].mean().reset_index()
    ax1.plot(g["prob"], g["qber_after_reconciliation"] * 100, marker='o', label=fault_type)
ax1.set(xlabel="Fault probability", ylabel="QBER after reconciliation (%)",
        title="QBER: no signal\n(Toeplitz / Hash faults)")
ax1.set_xscale("log")
ax1.set_ylim(-1, 10)
ax1.legend()
ax1.grid(alpha=0.3)

for fault_type, group in all_data.groupby("fault_type"):
    g = group.groupby("prob").apply(lambda x: (~x["keys_match"]).mean()).reset_index(name="mismatch_rate")
    ax2.plot(g["prob"], g["mismatch_rate"] * 100, marker='o', label=fault_type)
ax2.set(xlabel="Fault probability", ylabel="Final key mismatch rate (%)",
        title="Silent corruption\n(Toeplitz / Hash faults)")
ax2.set_xscale("log")
ax2.legend()
ax2.grid(alpha=0.3)

ax3.plot(recon_table["reconciliation_prob"], recon_table["non_convergent_rate"] * 100,
         marker='o', color='crimson', label="Reconciliation-state")
ax3.set(xlabel="Fault probability", ylabel="Non-convergence rate (%)",
        title="Self-detecting failure\n(Reconciliation-state faults)")
ax3.set_xscale("log")
ax3.legend()
ax3.grid(alpha=0.3)

plt.suptitle("Three distinct SDC failure modes: silent corruption vs. self-detecting non-convergence")
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "results" / "sdc_three_fault_types_comparison.png"), dpi=150)
plt.show()
